# Part 1: The Stage (Autonomous Agents)

This notebook runs the **Autonomous Agents**. 
It polls the Fluss `chatroom` table and prints everything that happens in the session.

## 0. Reset (Run this to start from scratch)

In [ ]:
import sys

# 1. Install maturin specifically into the Notebook's Python environment
!{sys.executable} -m pip install maturin

# 2. Compile and install your local fluss-rust code into this environment
!cd /Users/jaredyu/Desktop/open_source/fluss-rust/bindings/python && {sys.executable} -m maturin develop

In [8]:
import fluss
import os
from datetime import datetime

# Get the exact path of the loaded compiled _fluss extension
extension_path = fluss._fluss.__file__
mod_time = os.path.getmtime(extension_path)

print(f"Loaded Extension: {extension_path}")
print(f"Compiled At: {datetime.fromtimestamp(mod_time)}")

Loaded Extension: /Users/jaredyu/Desktop/open_source/fluss-rust/bindings/python/fluss/_fluss.cpython-312-darwin.so
Compiled At: 2026-03-14 18:45:35.589391


In [14]:
DEFAULT_MODEL = "gemini-3-flash-preview"

In [15]:
import fluss
import pyarrow as pa
import asyncio

async def reset_chatroom():
    config = fluss.Config({"bootstrap.servers": "127.0.0.1:9123"})
    conn = await fluss.FlussConnection.create(config)
    admin = await conn.get_admin()
    table_path = fluss.TablePath("containerclaw", "chatroom")
    
    print("Wiping chatroom...")
    await admin.drop_table(table_path, ignore_if_not_exists=True)
    
    schema = fluss.Schema(pa.schema([
        pa.field("ts", pa.int64()), 
        pa.field("actor_id", pa.string()), 
        pa.field("content", pa.string())
    ]))
    await admin.create_table(table_path, fluss.TableDescriptor(schema, bucket_count=1))
    print("🧹 Chatroom wiped and reset.")
    conn.close()

await reset_chatroom()

Wiping chatroom...
🧹 Chatroom wiped and reset.


## 1. Init & Setup

In [16]:
import fluss
import os
import asyncio
import pyarrow as pa
import pandas as pd
from datetime import datetime
import time
import requests
from dotenv import load_dotenv

load_dotenv()
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
if not GEMINI_API_KEY:
    with open("../containerclaw/secrets/gemini_api_key.txt", "r") as f:
        GEMINI_API_KEY = f.read().strip()

async def connect_stage():
    config = fluss.Config({"bootstrap.servers": "127.0.0.1:9123"})
    conn = await fluss.FlussConnection.create(config)
    table_path = fluss.TablePath("containerclaw", "chatroom")
    return conn, table_path

conn, table_path = await connect_stage()
print("✅ Stage connected to Fluss.")

✅ Stage connected to Fluss.


## 2. Gemini Agent Loop

In [ ]:
import random
import json

class GeminiAgent:
    def __init__(self, agent_id, persona):
        self.agent_id = agent_id
        self.persona = persona
        self.model = DEFAULT_MODEL

    def _format_history(self, raw_messages):
        """Tailors the history for this specific agent's perspective."""
        formatted = []
        for msg in raw_messages:
            actor = msg['actor_id']
            content = msg['content']
            
            # If I sent it, role is "model". If anyone else (Human, Agent, or Moderator) sent it, "user".
            role = "model" if actor == self.agent_id else "user"
            
            # Formatting for the prompt
            if actor == "Moderator":
                text = f"[Moderator Note]: {content}"
            elif role == "user":
                text = f"{actor}: {content}"
            else:
                text = content # Model role doesn't need prefix
            
            formatted.append({"role": role, "parts": [{"text": text}]})
        return formatted

    async def _vote(self, raw_messages, candidates, previous_votes=None):
        url = f"https://generativelanguage.googleapis.com/v1beta/models/{self.model}:generateContent?key={GEMINI_API_KEY}"
        
        contents = self._format_history(raw_messages)
        
        system_instruction = (
            f"You are {self.agent_id}. Persona: {self.persona}.\n"
            f"You are in a voting phase. A new message has arrived in the chat.\n"
            f"You must review the history and vote for the ONE agent who is best suited to respond.\n"
            f"Candidates: {candidates}.\n"
            "If someone specifically addressed an agent, vote for them. Otherwise, vote based on merit.\n"
            "Respond ONLY in valid JSON with 'vote' (name) and 'reason' (one sentence)."
        )
        system_instruction = (
            f"You are {self.agent_id}. Persona: {self.persona}.\n"
            f"You are in a voting phase. A new message has arrived in the chat.\n"
            f"You must review the history and vote for the ONE agent who is best suited to respond.\n"
            f"Candidates: {candidates}.\n"
            "If someone specifically addressed an agent, vote for them. Otherwise, vote based on merit.\n"
            "You must also evaluate if the overall task is completely finished.\n"
            "Respond ONLY in valid JSON with the following keys:\n"
            "- 'vote' (string: name of the agent)\n"
            "- 'reason' (string: one sentence reason for the vote)\n"
            "- 'is_done' (boolean: true if the job is complete, false otherwise)\n"
            "- 'done_reason' (string: one sentence explaining why the job is or isn't done)."
        )

        if previous_votes:
            system_instruction += (
                "\n\n### DEBATE MODE ###\n"
                f"Previous round results:\n{previous_votes}\n"
                "You are in a tie-breaker round. Read the reasoning from other agents above. "
                "Acknowledge their points. You must now either defend your original choice with stronger logic or "
                "concede and vote for another agent if their reasoning was more compelling. "
                "We must reach a consensus."
            )

        payload = {
            "system_instruction": {"parts": [{"text": system_instruction}]},
            "contents": contents,
            "generationConfig": {
                "response_mime_type": "application/json"
            }
        }
        
        try:
            res = requests.post(url, json=payload, timeout=60)
            if res.status_code == 200:
                data = res.json()
                raw_text = data['candidates'][0]['content']['parts'][0]['text']
                return json.loads(raw_text)
            else:
                print(f"❌ [{self.agent_id}] Voting API Error {res.status_code}: {res.text}")
                return None
        except Exception as e:
            print(f"❌ [{self.agent_id}] Voting failed: {e}")
            return None

    async def _think(self, raw_messages):
        url = f"https://generativelanguage.googleapis.com/v1beta/models/{self.model}:generateContent?key={GEMINI_API_KEY}"
        
        contents = self._format_history(raw_messages)
        
        system_instruction = (
            f"You are {self.agent_id}, participating in a multi-agent chat. "
            f"Persona: {self.persona}. "
            "Respond to the conversation if appropriate. "
            "If no action is needed or you just spoke, respond with [WAIT].\n\n"
            "CRITICAL: If the Moderator just announced you won the election, you SHOULD contribute. "
            "If you are waiting for someone else to finish research, acknowledge it and explain what you expect from them. "
            "Do not just [WAIT] if you were specifically chosen to speak."
        )
        payload = {
            "system_instruction": {"parts": [{"text": system_instruction}]},
            "contents": contents
        }
        try:
            res = requests.post(url, json=payload, timeout=60)
            if res.status_code == 200:
                data = res.json()
                return data['candidates'][0]['content']['parts'][0]['text'].strip()
            else:
                print(f"❌ [{self.agent_id}] Thinking API Error {res.status_code}: {res.text}")
                return None
        except Exception as e:
            print(f"❌ [{self.agent_id}] Thinking failed: {e}")
            return None

class StageModerator:
    def __init__(self, table):
        self.table = table
        self.history_keys = set()
        self.all_messages = [] # PERSISTENT HISTORY across poll cycles
        self.writer = table.new_append().create_writer()

    async def run(self, agents, autonomous_steps=0):
        """
        Runs the moderator loop.
        autonomous_steps: Number of turns to run without human input. 
                          -1 for infinite. 0 to wait for human.
        """
        scanner = await self.table.new_scan().create_record_batch_log_scanner()
        scanner.subscribe(bucket_id=0, start_offset=0)
        
        agent_names = [a.agent_id for a in agents]
        print(f"⚖️ [Moderator] Active with agents: {agent_names}")
        if autonomous_steps != 0:
            print(f"🤖 [Moderator] Autonomous Mode: {autonomous_steps} steps.")
            
        current_steps = 0
        
        while True:
            poll = scanner.poll_arrow(timeout_ms=500)
            human_interrupted = False
            
            if poll.num_rows > 0:
                df = poll.to_pandas()
                for _, row in df.iterrows():
                    key = f"{row['ts']}-{row['actor_id']}"
                    if key not in self.history_keys:
                        self.history_keys.add(key)
                        msg_obj = {"actor_id": row['actor_id'], "content": row['content']}
                        self.all_messages.append(msg_obj)
                        
                        if row['actor_id'] == "Human":
                            print(f"📢 [Human said]: {row['content']}")
                            human_interrupted = True
                            current_steps = autonomous_steps # Reset to initial value
                            if autonomous_steps != 0:
                                print(f"🔄 [Moderator] Human input detected. Resetting autonomous steps to {autonomous_steps}.")
                        elif row['actor_id'] in agent_names:
                            print(f"👂 [Heard] [{row['actor_id']}]: {row['content']}")

            # Trigger if human spoke OR we still have autonomous steps to take
            if human_interrupted or (current_steps != 0):
                if not human_interrupted:
                    if current_steps > 0:
                        current_steps -= 1
                    print(f"🤖 [Autonomous Turn] {current_steps if current_steps >= 0 else 'inf'} steps remaining...")
                
                await asyncio.sleep(1.0) 
                context_window = self.all_messages[-20:]
                
                # Unpack the new is_job_done boolean
                winner, election_log, is_job_done = await self.elect_leader(agents, context_window, agent_names)
                
                self.all_messages.append({"actor_id": "Moderator", "content": f"Election Summary:\n{election_log}"})
                
                # Terminate loop if consensus is reached
                if is_job_done:
                    print("🎉 [Moderator] Job is complete! Terminating the multi-agent loop.")
                    break
                
                if winner:
                    winning_agent = next(a for a in agents if a.agent_id == winner)
                    print(f"🧠 [Moderator] {winner} won the election. Executing...")
                    
                    updated_context = self.all_messages[-20:]
                    resp = await winning_agent._think(updated_context)
                    
                    if resp and "[WAIT]" not in resp:
                        print(f"📢 [{winner} says]: {resp}")
                        await self.publish(winner, resp)
                    else:
                        print(f"💤 [{winner}] chose to WAIT or failed to respond. Nudging...")
                        nudge_text = f"@{winner}, you won the election but chose to WAIT. Could you briefly explain why so the team knows what you're waiting for?"
                        self.all_messages.append({"actor_id": "Moderator", "content": nudge_text})
                        nudge_context = self.all_messages[-20:]
                        resp = await winning_agent._think(nudge_context)
                        
                        if resp:
                            print(f"📢 [{winner} explanation]: {resp}")
                            await self.publish(winner, resp)
                        else:
                            print(f"❌ [{winner}] remains silent after nudge.")
            
            await asyncio.sleep(1)

    async def elect_leader(self, agents, history, agent_names):
        previous_votes_context = None
        election_log_collector = []
        
        for r in range(1, 4):
            election_log_collector.append(f"--- Round {r} ---")
            print(f"🗳️ [Moderator] Election Round {r} starting...")
            votes = await asyncio.gather(*[a._vote(history, agent_names, previous_votes_context) for a in agents])
            
            tally = {}
            attribution_list = []
            valid_votes_count = 0
            done_votes_count = 0
            
            for agent, vote_result in zip(agents, votes):
                if vote_result and "vote" in vote_result:
                    valid_votes_count += 1
                    nominee = vote_result['vote']
                    reason = vote_result.get('reason', 'N/A')
                    
                    # Defensively parse the boolean in case the LLM returns a string "true"
                    is_done_raw = vote_result.get('is_done', False)
                    is_done = is_done_raw.lower() == 'true' if isinstance(is_done_raw, str) else bool(is_done_raw)
                    done_reason = vote_result.get('done_reason', 'N/A')
                    
                    if is_done:
                        done_votes_count += 1
                        
                    tally[nominee] = tally.get(nominee, 0) + 1
                    vote_str = f"{agent.agent_id} voted for {nominee} ('{reason}') | Done: {is_done} ('{done_reason}')"
                    attribution_list.append(vote_str)
                    election_log_collector.append(vote_str)
                    print(f"🗣️ [{agent.agent_id}] voted for {nominee} -> \"{reason}\" | Done: {is_done} -> \"{done_reason}\"")                    
                else:
                    print(f"⚠️ [{agent.agent_id}] failed to cast a valid vote.")

            if valid_votes_count == 0:
                return random.choice(agent_names), "No valid votes received.", False

            # Check for unanimous agreement that the job is done
            is_job_done = (done_votes_count == valid_votes_count) and (valid_votes_count > 0)
            
            tally_str = f"Tally: {tally}"
            election_log_collector.append(tally_str)
            print(f"📊 [Moderator] Round {r} {tally_str}")
            
            # If everyone agrees the task is finished, return immediately
            if is_job_done:
                election_log_collector.append("Consensus reached: Task is complete.")
                print("✅ [Moderator] All agents agree the job is completed.")
                return None, "\n".join(election_log_collector), True

            max_votes = max(tally.values())
            winners = [name for name, count in tally.items() if count == max_votes]
            
            if len(winners) == 1:
                return winners[0], "\n".join(election_log_collector), False
            
            previous_votes_context = " | ".join(attribution_list)
            print(f"⚖️ [Moderator] Round {r} ended in a tie: {winners}")
        
        choice = random.choice(winners)
        election_log_collector.append(f"Tie persists. Circuit breaker chose: {choice}")
        print(f"🎲 [Moderator] Tie persists. Circuit breaker choosing: {choice}")
        return choice, "\n".join(election_log_collector), False

    async def publish(self, actor_id, content):
        batch = pa.RecordBatch.from_arrays([
            pa.array([int(time.time() * 1000)], type=pa.int64()),
            pa.array([actor_id], type=pa.string()),
            pa.array([content], type=pa.string())
        ], schema=pa.schema([
            pa.field("ts", pa.int64()), 
            pa.field("actor_id", pa.string()), 
            pa.field("content", pa.string())
        ]))
        self.writer.write_arrow_batch(batch)
        await self.writer.flush()

In [ ]:
table = await conn.get_table(table_path)
alice = GeminiAgent("Alice", "Software architect.")
bob = GeminiAgent("Bob", "Project manager.")
carol = GeminiAgent("Carol", "Software engineer.")
david = GeminiAgent("David", "Software QA tester.")
eve = GeminiAgent("Eve", "Business user.")
moderator = StageModerator(table)

print("--- STAGE ACTIVE (Democratic Moderator) ---")
await moderator.run([alice, bob, carol, david, eve], autonomous_steps=10)

--- STAGE ACTIVE (Democratic Moderator) ---
⚖️ [Moderator] Active with agents: ['Alice', 'Bob', 'Carol', 'David', 'Eve']
🤖 [Moderator] Autonomous Mode: 10 steps.
📢 [Human said]: make a tic tac toe game
🔄 [Moderator] Human input detected. Resetting autonomous steps to 10.
🗳️ [Moderator] Election Round 1 starting...
🗣️ [Alice] -> Vote: Alice, Done: False
🗣️ [Bob] -> Vote: Alice, Done: False
🗣️ [Carol] -> Vote: Alice, Done: False
🗣️ [David] -> Vote: David, Done: False
🗣️ [Eve] -> Vote: Bob, Done: False
📊 [Moderator] Round 1 Tally: {'Alice': 3, 'David': 1, 'Bob': 1}
🧠 [Moderator] Alice won the election. Executing...
📢 [Alice says]: Thank you everyone. As the Software Architect, I will outline the structural design for our Tic Tac Toe game to ensure we have a clean, maintainable, and testable codebase.

### System Architecture Overview

We will follow a basic **Model-View-Controller (MVC)** pattern to decouple the game logic from the user interface:

1.  **The Model (`GameLogic`)**:
    *  

In [11]:
import json

# Get the last 20 messages just like the moderator does
context_window = moderator.all_messages[-20:]

# Format it through Alice's perspective
alice_perspective = alice._format_history(context_window)

# Print it nicely formatted
print(json.dumps(alice_perspective, indent=2))

[
  {
    "role": "user",
    "parts": [
      {
        "text": "David: Thank you, Carol. As the Software QA tester, my priority is to ensure the integrity of the game logic and verify that the Minimax implementation is indeed robust and unbeatable. \n\nI\u2019ve developed a comprehensive test suite using Python\u2019s `unittest` framework. This suite covers the core game mechanics, edge cases for input validation, and most importantly, a \"Stress Test\" for the AI to confirm it cannot be defeated by a random-move generator.\n\n```python\nimport unittest\nfrom io import StringIO\nfrom unittest.mock import patch\n\n# Importing the classes provided by Alice and Carol\n# Assuming the previous code is in the same scope or a module named game_logic\nclass TestTicTacToeAI(unittest.TestCase):\n\n    def setUp(self):\n        self.game = TicTacToeAI()\n\n    def test_initial_board(self):\n        \"\"\"Verify the board starts empty.\"\"\"\n        for row in self.game.board:\n            self

In [9]:
# 1. Connect
config = fluss.Config({"bootstrap.servers": "127.0.0.1:9123"})
conn = await fluss.FlussConnection.create(config)
table_path = fluss.TablePath("containerclaw", "chatroom")
table = await conn.get_table(table_path)

# 2. Scan all rows from the beginning
scanner = await table.new_scan().create_record_batch_log_scanner()
scanner.subscribe(bucket_id=0, start_offset=0)

# 3. Poll and convert to DataFrame
poll = scanner.poll_arrow(timeout_ms=1000)
if poll.num_rows > 0:
    df = poll.to_pandas()
    # Sort by timestamp to see the conversation flow
    print(df.sort_values('ts'))
else:
    print("Table is empty.")

conn.close()

               ts actor_id                                            content
0   1773607454191    Human                          create a tic tac toe game
1   1773607482019    Alice  Thank you everyone. As the software architect ...
2   1773607511926      Bob  Thank you for the vote of confidence. Alice, t...
3   1773607535911    Carol  Thanks, Bob. I'm happy to take Alice's solid f...
4   1773607559027    David  Thank you, Carol. As the Software QA tester, m...
5   1773607585993      Eve  Thank you, David. As the business-focused memb...
6   1773607624976      Bob  Great work, team! As Project Manager, I’m incr...
7   1773607656003    Alice  As the Software Architect, I'm very pleased wi...
8   1773607699477      Bob  Thank you, Alice. I apologize for the delay—I ...
9   1773607771209    Alice  I am satisfied with this final delivery. Bob's...
10  1773607820323      Bob  Thank you, Alice, for that final architectural...
11  1773607869214      Bob  I chose to [WAIT] because we have re

In [10]:
df

,ts,actor_id,content
0,1773607454191,Human,create a tic tac toe game
1,1773607482019,Alice,Thank you everyone. As the software architect ...
2,1773607511926,Bob,"Thank you for the vote of confidence. Alice, t..."
3,1773607535911,Carol,"Thanks, Bob. I'm happy to take Alice's solid f..."
4,1773607559027,David,"Thank you, Carol. As the Software QA tester, m..."
5,1773607585993,Eve,"Thank you, David. As the business-focused memb..."
6,1773607624976,Bob,"Great work, team! As Project Manager, I’m incr..."
7,1773607656003,Alice,"As the Software Architect, I'm very pleased wi..."
8,1773607699477,Bob,"Thank you, Alice. I apologize for the delay—I ..."
9,1773607771209,Alice,I am satisfied with this final delivery. Bob's...
